# 00 · Ortam ve veri kontrolü
Çalıştırma sırası: **00 → 01 → 02 → 03 → 04 → 05 → 06**. Her notebook kaldığı yerden devam eder (tamamlanan koşular atlanır).
Önce `SMOKE = True` ile hepsini bir kez çalıştırıp kurulumu doğrulamak iyi olur (birkaç dakika); sonra `results\` klasörünü silip `SMOKE = False`.

In [ ]:
import importlib, platform
for pkg in ['torch', 'numpy', 'pandas', 'scipy', 'matplotlib', 'openpyxl', 'mat73', 'pyarrow']:
    try:
        m = importlib.import_module(pkg); print(f'✅ {pkg:10s} {getattr(m, "__version__", "")}')
    except Exception as e:
        print(f'❌ {pkg:10s} yok → conda env update -f environment.yml  (ya da pip install {pkg})')
import torch
print('\nCUDA kullanılabilir:', torch.cuda.is_available(), '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')

In [ ]:
import sys, os, json, time, warnings
import os; ROOT = os.path.abspath('..')  # repository root (run notebooks from the notebooks/ folder)
SMOKE = False                 # True: çok küçük hızlı deneme (kurulum testi)    False
sys.path.insert(0, ROOT)
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, torch
from data_utils import *
from models.experiment import *
from models import engine
exp = Experiment(ROOT, smoke=SMOKE)
exp.load_data()
print('Cihaz:', exp.dev, '| W =', exp.cfg['W'])

In [ ]:
info = exp.env_info(); print(json.dumps(info, indent=1, ensure_ascii=False))
wc = window_counts(exp.role_dfs, exp.te_dfs, TRAIN_ROLES, VAL_ROLES, exp.cfg['W'])
print(wc.to_string(index=False))
with pd.ExcelWriter(exp.p['excel'] / '00_veri.xlsx') as xw:
    exp.file_table.to_excel(xw, sheet_name='Dosyalar', index=False)
    wc.to_excel(xw, sheet_name='Pencere sayıları', index=False)
    pd.DataFrame([{'ön işleme (ms / 1 Hz örnek)': exp.pre_ms}]).to_excel(xw, sheet_name='Ön işleme', index=False)
print(exp.file_table.round(3).to_string(index=False))

In [ ]:
# Hızlı ileri geçiş testi (tüm modeller)
ds = exp.datasets(exp.cfg['W']); X, y = next(ds['train'].batches(8, False))
for name, spec in BASE_SPECS.items():
    m = exp.model_for(spec); p, _ = m(X); print(f'{name:28s} çıktı {tuple(p.shape)} | parametre {sum(q.numel() for q in m.parameters()):,}')